|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Detokenization<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: the streaming detokenizer<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
print('vocab', tok.vocab_size)

Write the streaming detokenizer.

No GPU, no tensors, and no interesting arithmetic. This stage is boring and it
is the source of most user-visible bugs in real servers, which is a fair
trade for one afternoon.

# Exercise 1: emit the difference, not the token

A character can be split across two tokens, so decoding one token at a time
produces replacement characters. Decode the prefix instead and emit whatever
is new.

In [ ]:
class Incremental:
  def __init__(self, tokenizer):
    self.tok = tokenizer; self.ids = []; self.emitted = 0

  def push(self, token_id):
    self.ids.append(token_id)
    text = self.tok.decode(self.ids)
    if text.endswith('\ufffd'):
      return ''
    out = text[self.emitted:]
    self.emitted = len(text)
    return out

  def flush(self):
    """The stream ended. Emit whatever is left, broken or not."""
    text = self.tok.decode(self.ids)
    out, self.emitted = text[self.emitted:], len(text)
    return out

ids = tok('hello world', add_special_tokens=False).input_ids
d = Incremental(tok)
print([d.push(i) for i in ids])

# Exercise 2: prove it with a fuzz test

The contract is an equality. Test it that way.

In [ ]:
import random

rng = random.Random(0)
fails = 0
for _ in range(500):
  ids = [rng.randrange(tok.vocab_size) for _ in range(rng.randint(1, 40))]
  d = Incremental(tok)
  if ''.join(d.push(i) for i in ids) + d.flush() != tok.decode(ids):
    fails += 1
print(f'{fails} failures in 500 random sequences')

hard = ['\U0001F468\u200d\U0001F469\u200d\U0001F467\u200d\U0001F466',
        '\U0001F3F3\ufe0f\u200d\U0001F308', 'caf\u00e9', '\u65e5\u672c\u8a9e',
        '\uc548\ub155\ud558\uc138\uc694']
for t in hard:
  ids = tok(t, add_special_tokens=False).input_ids
  d = Incremental(tok)
  ok = ''.join(d.push(i) for i in ids) + d.flush() == tok.decode(ids)
  print(f'{ok}  {t!r}')

# Exercise 3: stop strings that straddle

The model emits ` EN` then `D`. Neither token contains `END`.

In [ ]:
def stream_until_stop(ids, stop):
  d, out, hold = Incremental(tok), [], ''
  for i in ids:
    hold += d.push(i)
    pos = hold.find(stop)
    if pos >= 0:
      out.append(hold[:pos]); return ''.join(out), True
    keep = 0
    for k in range(1, min(len(stop), len(hold))):
      if hold.endswith(stop[:k]): keep = k
    out.append(hold[:len(hold)-keep] if keep else hold)
    hold = hold[len(hold)-keep:] if keep else ''
  return ''.join(out) + hold + d.flush(), False

cases = [('Answer: yes. END OF LINE', 'END'),
         ('nothing to stop for', 'END'),
         ('the ENDING is near', 'END')]
for text, stop in cases:
  ids = tok(text, add_special_tokens=False).input_ids
  emitted, stopped = stream_until_stop(ids, stop)
  print(f'{text!r}\n  -> {emitted!r}  stopped={stopped}')

### The third case is the one that matters

`'the ENDING is near'` contains `END`, so the stream stops after `the `.
That is correct, and it will still surprise somebody: a stop string is a
substring match, not a word match, and users who set `stop=['\\n']` are
sometimes astonished by what counts as a newline.

Three properties your implementation has, and none of them are optional:

- **It never emits a prefix of the stop string.** Otherwise `EN` appears
  in the user's browser for one frame and then the stream ends.
- **It decodes prefixes, not tokens.** So a character split across two
  tokens is never shown as a replacement character.
- **It is covered by an equality, not by examples.** Streamed output must
  equal batch-decoded output, for any token sequence at all.

This stage has no interesting arithmetic and no kernel. It is worth doing
carefully anyway, because these are the bugs users actually see, and
because a fuzz test on a pure function is the cheapest correctness you
will ever buy.

    ./vc guide 14